# Intelligent Complaint Triage using Fine-Tuned Qwen2.5-7B

## Problem

Financial institutions receive thousands of customer complaints in free-form natural language.

Before a complaint can be investigated, operations teams must manually determine:

- Product
- Sub-product
- Issue
- Sub-issue

Manual triaging is slow, inconsistent, and difficult to scale.

## Solution

A domain-adapted Qwen2.5-7B model was fine-tuned on CFPB complaint taxonomy data to automatically classify complaints into the correct operational categories.

This demo compares:

1. Base Qwen2.5-7B-Instruct
2. Fine-Tuned CFPB Model

on real complaint scenarios from the evaluation dataset.

## 1. Install Dependencies

In [1]:
# Run once -- packages are pinned to versions used during training
!pip install -q \
    transformers==4.44.0 \
    peft==0.12.0 \
    accelerate==0.34.0 \
    scikit-learn



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 2. Environment & GPU Check

In [2]:
import torch

print("=" * 55)
print("  ENVIRONMENT")
print("=" * 55)
print(f"  PyTorch version    : {torch.__version__}")
print(f"  ROCm/HIP available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    total_vram_gb = props.total_memory / 1e9
    print(f"  GPU device         : {torch.cuda.get_device_name(0)}")
    print(f"  Total VRAM         : {total_vram_gb:.1f} GB")
else:
    print("  WARNING: No GPU detected.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  Active device      : {DEVICE}")
print("=" * 55)


  ENVIRONMENT
  PyTorch version    : 2.10.0+rocm7.2.4.git3d3aa833
  ROCm/HIP available : True
  GPU device         : AMD Instinct MI300X
  Total VRAM         : 206.1 GB
  Active device      : cuda


## 3. Load Base Model

We load the unmodified `Qwen2.5-7B-Instruct` in bfloat16 -- the same precision
used during training. No quantisation is applied; this is a clean ROCm-native load.

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_PATH    = "/workspace/shared/Day4/models/qwen2.5-7b-lora"
MAX_NEW_TOKENS  = 128


def gpu_memory_snapshot(label):
    """Print current GPU memory allocation."""
    if not torch.cuda.is_available():
        return
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved  = torch.cuda.memory_reserved(0)  / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  [{label}]")
    print(f"    Allocated : {allocated:.2f} GB")
    print(f"    Reserved  : {reserved:.2f} GB")
    print(f"    Total     : {total:.1f} GB")


torch.cuda.reset_peak_memory_stats()
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model (bfloat16)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
base_model.eval()
print(f"Base model loaded. Parameters: {base_model.num_parameters():,}")
print()
gpu_memory_snapshot("After base model load")


Loading tokenizer...
Loading base model (bfloat16)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Base model loaded. Parameters: 7,615,616,512

  [After base model load]
    Allocated : 15.75 GB
    Reserved  : 15.98 GB
    Total     : 206.1 GB


## 4. Base Model Inference -- Before Fine-Tuning

The unmodified Qwen2.5-7B-Instruct has not seen the CFPB taxonomy.
It will attempt to answer but typically produces free-form text, incorrect
field names, or an entirely wrong format -- none of which can be ingested
by a complaint management system.

In [6]:
import time

# Two real complaint scenarios used throughout this demo
SCENARIO_1 = (
"""
On XXXX XXXX, 2015, my mother passed away I was named Executrix of the estate. 
On XXXX XXXX, 2015, I mailed a copy of the death certificate, letter of Testamentary, and letter of intent to sell the property to Wells Fargo to stop the reverse mortgage. 
On XXXX XXXX, 2015, the appraisal ordered by Wells Fargo was completed. On XXXX XXXX, 2015, I contacted the representative from Wells Fargo that was provided to me.
The paperwork was received and I inquired as to the results of the appraisal. She stated that she had not received it and to call back in one week. On XXXX XXXX, XXXX, 
I again contacted Wells Fargo regarding the status of the appraisal. I was told there was no appraisal had been submitted to Wells Fargo but she would check into. 
She advised to use XXXX to establish a selling price. On XXXX XXXX, 2015, I left a message for the Wells Fargo and to date there has been no follow-up from Wells Fargo. On XXXX XXXX, 2015,
I called Wells Fargo to obtain an email address to submit my concerns. I was advised that all communication can only be through phone or fax. On XXXX XXXX, 2015,
I faxed a letter demanding a copy of the appraisal by XXXX XXXX, 2015 or I would contact your organization. To date there has been no response from Wells Fargo regarding my request.
"""
    
)

SCENARIO_2 = (
"""
I contacted Shell Citibank XXXX XXXX. I called customer service. I asked them about a credit limit increase if it was a soft or a hard inquiry. 
I explained I was going to go for a major purchase and I ca n't have any hard inquiries on my credit. They said they could not answer this question. 
They said since you have a fraud alert to fill it out online and they will contact me before they pull the credit whether its soft or hard. This did not happen it was a hard pull the next day. 
I spoke to several supervisors the XXXX XXXX I spoke with said it would take up to 10 days to remove it. This did not happen. I dealt with citi executive response I spoke to a woman named XXXX XXXX she can be reached at XXXX. 
She did nothing at her level to resolve the issue. She said because I filled out online with my income what I was told to fill out by customer service. That I authorized my credit to be ran. She did nothing to address the negligence and incompetence of the company representative misinforming me. 
She does not even care I might be denied for my large purchase because of this inquiry. I think this is a very dirty tactic that they play. I have allready filed a complaint with the ftc also.
I am sick and tired of companys not being held accountable by people who work for them and misinform customers. Everything was all done over the phone. 
My credit was pulled once through XXXX. XXXX no longer allows you to dispute through the credit bureau only with the creditor.
"""
)


def build_messages(complaint_text):
    """Wrap complaint text in the chat format used during training."""
    return [
        {
            "role": "system",
            "content": (
                "You are a banking complaint classification assistant. "
                "Given a consumer complaint narrative, extract the CFPB ticket fields "
                "as a JSON object with keys: product, sub_product, issue, sub_issue."
            ),
        },
        {"role": "user", "content": complaint_text},
    ]


def run_inference(complaint_text, model, label):
    """
    Run one inference pass and return output text plus profiling stats:
    input token count, output token count, latency, peak GPU memory.
    """
    messages = build_messages(complaint_text)
    prompt   = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs    = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    input_len = inputs["input_ids"].shape[1]

    torch.cuda.reset_peak_memory_stats()
    t_start = time.time()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    latency_ms  = (time.time() - t_start) * 1000
    output_len  = output.shape[1] - input_len
    peak_mem_gb = torch.cuda.max_memory_allocated(0) / 1e9
    generated   = tokenizer.decode(output[0][input_len:], skip_special_tokens=True)

    return {
        "label"        : label,
        "output"       : generated,
        "input_tokens" : input_len,
        "output_tokens": output_len,
        "total_tokens" : input_len + output_len,
        "latency_ms"   : round(latency_ms, 1),
        "peak_gpu_gb"  : round(peak_mem_gb, 2),
    }


def print_result(result, complaint):
    """Print a clean inference result summary."""
    print("=" * 62)
    print(f"  {result['label']}")
    print("=" * 62)
    print(f"  Complaint  : {complaint[:95]}...")
    print(f"  Output     :")
    print(f"    {result['output'].strip()}")
    print()
    print(f"  Input tokens  : {result['input_tokens']}")
    print(f"  Output tokens : {result['output_tokens']}")
    print(f"  Total tokens  : {result['total_tokens']}")
    print(f"  Latency       : {result['latency_ms']} ms")
    print(f"  Peak GPU mem  : {result['peak_gpu_gb']} GB")
    print("=" * 62)
    print()


print("--- Scenario 1: Mortgage ---")
base_s1 = run_inference(SCENARIO_1, base_model, "BASE MODEL | Scenario 1 (Mortgage)")
print_result(base_s1, SCENARIO_1)

print("--- Scenario 2: Credit card ---")
base_s2 = run_inference(SCENARIO_2, base_model, "BASE MODEL | Scenario 2 (Credit card)")
print_result(base_s2, SCENARIO_2)


--- Scenario 1: Mortgage ---
  BASE MODEL | Scenario 1 (Mortgage)
  Complaint  : 
On XXXX XXXX, 2015, my mother passed away I was named Executrix of the estate. 
On XXXX XXXX, ...
  Output     :
    ```json
{
  "product": "Reverse Mortgage",
  "sub_product": "Home Equity Conversion Mortgage (HECM)",
  "issue": "Communication Issues",
  "sub_issue": "Failure to Respond to Customer Requests"
}
```

  Input tokens  : 382
  Output tokens : 50
  Total tokens  : 432
  Latency       : 1198.2 ms
  Peak GPU mem  : 16.2 GB

--- Scenario 2: Credit card ---
  BASE MODEL | Scenario 2 (Credit card)
  Complaint  : 
I contacted Shell Citibank XXXX XXXX. I called customer service. I asked them about a credit l...
  Output     :
    ```json
{
  "product": "Credit reporting",
  "sub_product": "Credit card",
  "issue": "Inaccurate or incomplete information",
  "sub_issue": "Hard credit inquiry without customer's knowledge"
}
```

  Input tokens  : 382
  Output tokens : 50
  Total tokens  : 432
  Latency  

## 5. Load LoRA Adapter -- Fine-Tuned Model

The LoRA adapter is attached on top of the already-loaded base model.
No additional VRAM is needed for the base weights -- only the small adapter matrices
are newly allocated (~1% of total parameters).

**Training configuration:**

| Parameter | Value |
|-----------|-------|
| LoRA rank (r) | 16 |
| LoRA alpha | 32 |
| Target modules | q_proj, k_proj, v_proj, o_proj |
| Epochs | 5 (early stopping, patience=3) |
| Effective batch size | 32 (8 per device x 4 grad accum) |
| Learning rate | 1e-4 |
| Dataset | Full CFPB training split |
| Training time | ~45 minutes |
| Hardware | AMD Instinct MI300X, 192 GB VRAM |


In [7]:
from peft import PeftModel

print("Attaching LoRA adapter...")
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
ft_model.eval()
print("Adapter loaded.")
print()
gpu_memory_snapshot("After adapter load")


Attaching LoRA adapter...
Adapter loaded.

  [After adapter load]
    Allocated : 15.87 GB
    Reserved  : 16.54 GB
    Total     : 206.1 GB


## 6. Fine-Tuned Model Inference -- After LoRA Adaptation

Same two complaints. The fine-tuned model now produces structured JSON
aligned to the CFPB taxonomy, directly consumable by a complaint management system.

In [8]:
print("--- Scenario 1: Mortgage ---")
ft_s1 = run_inference(SCENARIO_1, ft_model, "FINE-TUNED | Scenario 1 (Mortgage)")
print_result(ft_s1, SCENARIO_1)

print("--- Scenario 2: Credit Card ---")
ft_s2 = run_inference(SCENARIO_2, ft_model, "FINE-TUNED | Scenario 2 (Credit Card)")
print_result(ft_s2, SCENARIO_2)


--- Scenario 1: Mortgage ---
  FINE-TUNED | Scenario 1 (Mortgage)
  Complaint  : 
On XXXX XXXX, 2015, my mother passed away I was named Executrix of the estate. 
On XXXX XXXX, ...
  Output     :
    {"product": "Mortgage", "sub_product": "Reverse mortgage", "issue": "Settlement process and costs", "sub_issue": "Not specified"}

  Input tokens  : 382
  Output tokens : 35
  Total tokens  : 417
  Latency       : 6765.3 ms
  Peak GPU mem  : 16.24 GB

--- Scenario 2: Credit Card ---
  FINE-TUNED | Scenario 2 (Credit Card)
  Complaint  : 
I contacted Shell Citibank XXXX XXXX. I called customer service. I asked them about a credit l...
  Output     :
    {"product": "Credit card", "sub_product": "General-purpose credit card or charge card", "issue": "Improper use of your report", "sub_issue": "Credit inquiries on your report"}

  Input tokens  : 382
  Output tokens : 43
  Total tokens  : 425
  Latency       : 1661.9 ms
  Peak GPU mem  : 16.24 GB



## 7. Side-by-Side Comparison

LoRA adds no inference overhead -- the adapter weights are absorbed into the base
model layers at load time. Latency and memory are identical; only output quality changes.

In [10]:
print("=" * 70)
print("  OUTPUT COMPARISON -- BEFORE vs AFTER FINE-TUNING")
print("=" * 70)

pairs = [
    ("Scenario 1 -- Mortgage",         SCENARIO_1, base_s1, ft_s1),
    ("Scenario 2 -- Credit Card", SCENARIO_2, base_s2, ft_s2),
]

for name, complaint, base_r, ft_r in pairs:
    print(f"\n  {name}")
    print(f"  Complaint : {complaint[:95]}...")
    print()
    print("  BASE MODEL OUTPUT:")
    print(f"    {base_r['output'].strip()}")
    print()
    print("  FINE-TUNED OUTPUT:")
    print(f"    {ft_r['output'].strip()}")
    print("  " + "-" * 65)

print()
print("=" * 70)
print("  PROFILING SUMMARY")
print("=" * 70)
header = f"  {'Metric':<26}  {'S1 Base':>9}  {'S1 FT':>9}  {'S2 Base':>9}  {'S2 FT':>9}"
print(header)
print("  " + "-" * 66)

rows = [
    ("Input tokens",   "input_tokens"),
    ("Output tokens",  "output_tokens"),
    ("Total tokens",   "total_tokens"),
    ("Latency (ms)",   "latency_ms"),
    ("Peak GPU (GB)",  "peak_gpu_gb"),
]
for label, key in rows:
    print(
        f"  {label:<26}"
        f"  {base_s1[key]:>9}"
        f"  {ft_s1[key]:>9}"
        f"  {base_s2[key]:>9}"
        f"  {ft_s2[key]:>9}"
    )
print("=" * 70)


  OUTPUT COMPARISON -- BEFORE vs AFTER FINE-TUNING

  Scenario 1 -- Mortgage
  Complaint : 
On XXXX XXXX, 2015, my mother passed away I was named Executrix of the estate. 
On XXXX XXXX, ...

  BASE MODEL OUTPUT:
    ```json
{
  "product": "Reverse Mortgage",
  "sub_product": "Home Equity Conversion Mortgage (HECM)",
  "issue": "Communication Issues",
  "sub_issue": "Failure to Respond to Customer Requests"
}
```

  FINE-TUNED OUTPUT:
    {"product": "Mortgage", "sub_product": "Reverse mortgage", "issue": "Settlement process and costs", "sub_issue": "Not specified"}
  -----------------------------------------------------------------

  Scenario 2 -- Credit Card
  Complaint : 
I contacted Shell Citibank XXXX XXXX. I called customer service. I asked them about a credit l...

  BASE MODEL OUTPUT:
    ```json
{
  "product": "Credit reporting",
  "sub_product": "Credit card",
  "issue": "Inaccurate or incomplete information",
  "sub_issue": "Hard credit inquiry without customer's knowledge"


## 8. Full GPU Memory Snapshot

In [11]:
import subprocess

print("Current GPU state:")
try:
    result = subprocess.run(
        ["amd-smi", "monitor", "-m", "-u", "-t"],
        capture_output=True, text=True, timeout=10
    )
    print(result.stdout)
except Exception:
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved  = torch.cuda.memory_reserved(0)  / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU    : {torch.cuda.get_device_name(0)}")
    print(f"  Total  : {total:.1f} GB")
    print(f"  In use : {allocated:.2f} GB allocated / {reserved:.2f} GB reserved")


Current GPU state:
GPU  XCP   GPU_T   MEM_T   GFX_CLK   GFX%   MEM%  MEM_CLOCK
  0    0   53 °C   48 °C  2099 MHz    0 %    0 %    900 MHz



## 9. Final Results Summary

| Category                             | Base Qwen2.5-7B           | Fine-Tuned CFPB Model       | Improvement |
| ------------------------------------ | ------------------------- | --------------------------- | ----------- |
| Product Classification (Exact Match) | 1.0%                      | 90.8%                       | +89.8 pts   |
| Product F1 Score                     | 1.96%                     | 90.7%                       | +88.7 pts   |
| Sub-Product ROUGE-L                  | 0.004                     | 0.712                       | +0.708      |
| Issue ROUGE-L                        | 0.002                     | 0.401                       | +0.399      |
| Sub-Issue ROUGE-L                    | 0.000                     | 0.521                       | +0.521      |
| Output Structure                     | Inconsistent / Unreliable | Valid CFPB Taxonomy JSON    | ✓           |
| Taxonomy Alignment                   | Poor                      | High                        | ✓           |
| Manual Review Required               | High                      | Significantly Reduced       | ✓           |
| Additional Parameters Trained        | 0                         | LoRA Adapters Only          | Efficient   |
| Training Time                        | —                         | ~45 Minutes                 | Fast        |
| Inference Latency                    | Baseline                  | Near Identical              | No Impact   |
| Additional GPU Memory at Inference   | Baseline                  | Negligible (~50 MB Adapter) | No Impact   |

### Key Outcomes

* Product classification improved from **1.0% → 90.8% Exact Match**.
* Fine-tuning successfully learned the CFPB complaint taxonomy without modifying the base model weights.
* The model now produces structured, operationally useful complaint classifications across:

  * Product
  * Sub-Product
  * Issue
  * Sub-Issue
* LoRA adapters achieved substantial quality gains while maintaining nearly identical inference speed and memory footprint.
* A single inference call can convert an unstructured customer complaint into taxonomy-aligned JSON suitable for downstream routing and triage workflows.
